# Fine-tuning YOLOv9 on VisDrone (Aerial Imagery)

This tutorial shows how to fine-tune LibreYOLO's YOLOv9 detector on [VisDrone2019-DET](http://aiskyeye.com/), the standard benchmark for drone-captured object detection (10 classes, aerial perspective). VisDrone is a noticeably different data distribution than COCO — small objects, top-down view — so fine-tuning matters.

We walk through:
1. The VisDrone → YOLO annotation format conversion (a 10-line function).
2. Preparing a `data.yaml`.
3. Loading a COCO-pretrained YOLOv9 and fine-tuning on a small sample so this notebook runs on CPU.
4. Running inference and publishing to the HF Hub.

For a real training run, use `scripts/finetune_yolo9_visdrone.py`.

## 1. Install & imports

In [ ]:
# pip install -e git+https://github.com/aalvsz/libreyolo@agentic/c-visdrone-finetune#egg=libreyolo
import sys
from pathlib import Path
import numpy as np
import torch
import yaml
from PIL import Image

REPO = Path.cwd().parent if (Path.cwd().parent / 'scripts' / 'finetune_yolo9_visdrone.py').exists() else Path.cwd()
sys.path.insert(0, str(REPO / 'scripts'))

from finetune_yolo9_visdrone import (
    VISDRONE_CLASSES,
    visdrone_line_to_yolo,
    build_yolo_dataset,
)
from libreyolo.models.yolo9.model import LibreYOLO9
from libreyolo.models.yolo9.nn import LibreYOLO9Model

print('VisDrone classes:', VISDRONE_CLASSES)

## 2. The VisDrone annotation format

VisDrone stores one `.txt` per image under `annotations/`. Each line:

```
bbox_left, bbox_top, bbox_width, bbox_height, score, category, truncation, occlusion
```

Category IDs are 0–11. We drop 0 (ignored-regions) and 11 (others) and remap 1–10 → 0–9 so YOLO can use them directly.

`visdrone_line_to_yolo()` does exactly that, plus edge-bbox clamping and malformed-line handling:

In [ ]:
# Valid pedestrian (category 1) in a 1920x1080 image
print(visdrone_line_to_yolo('100,200,50,80,1,1,0,0', img_w=1920, img_h=1080))

# Ignored-region (category 0) → None
print(visdrone_line_to_yolo('10,10,20,20,1,0,0,0', img_w=1920, img_h=1080))

# Malformed → None
print(visdrone_line_to_yolo('garbage', img_w=1920, img_h=1080))

## 3. Build a tiny fake VisDrone dataset

The real dataset is ~3 GB. For this notebook we synthesize a 4-image dataset in the VisDrone directory layout, then run the conversion. The same `build_yolo_dataset()` function works on the real dataset unchanged.

In [ ]:
ROOT = Path('demo_visdrone').resolve()
if ROOT.exists():
    import shutil; shutil.rmtree(ROOT)

for split_suffix in ('DET-train', 'DET-val'):
    split_dir = ROOT / f'VisDrone2019-{split_suffix}'
    (split_dir / 'images').mkdir(parents=True)
    (split_dir / 'annotations').mkdir(parents=True)
    for i in range(2):
        rng = np.random.default_rng(hash((split_suffix, i)) % (2**32))
        img = rng.integers(40, 130, size=(480, 640, 3), dtype=np.uint8)
        img[200:300, 250:400] = 220
        Image.fromarray(img).save(split_dir / 'images' / f'{i}.jpg', quality=85)
        (split_dir / 'annotations' / f'{i}.txt').write_text(
            '250,200,150,100,1,4,0,0\n'   # one 'car' bbox
            '10,10,20,20,1,0,0,0\n'       # ignored region — will be dropped
        )

yolo_root = ROOT / 'yolo_fmt'
data_yaml = build_yolo_dataset(ROOT, yolo_root)
print('data.yaml:', data_yaml)
print(yaml.safe_load(data_yaml.read_text()))

## 4. Fine-tune

In production: `LibreYOLO9('LibreYOLO9s.pt', size='s')` auto-downloads the COCO-pretrained backbone, then `.train()` fine-tunes on the VisDrone labels. Here we use a fresh random checkpoint to keep the notebook offline.

In [ ]:
# Offline: random-init YOLOv9-t so the notebook needs no network
init_ckpt = ROOT / 'yolo9t-init.pt'
torch.save({'model': LibreYOLO9Model(config='t', nb_classes=10).state_dict()}, init_ckpt)

model = LibreYOLO9(model_path=str(init_ckpt), size='t', nb_classes=10, device='cpu')

results = model.train(
    data=str(data_yaml),
    epochs=2, batch=2, imgsz=128, lr0=0.001, optimizer='SGD',
    device='cpu', workers=0, amp=False, patience=2,
    project=str(ROOT / 'runs'), name='visdrone_demo', exist_ok=True,
)
print('final_loss:', results['final_loss'])
print('best_ckpt:', results['best_checkpoint'])

## 5. Inference

With a real training run, this is where you'd see per-class aerial detections. With our 2-epoch random-init demo, results will be arbitrary — focus on the API.

In [ ]:
sample = yolo_root / 'images' / 'val' / '0.jpg'
result = model(sample, conf=0.001, iou=0.5)
res = result[0] if isinstance(result, list) else result
print('num detections:', len(res.boxes))
if len(res.boxes):
    print('boxes (xyxy):', res.boxes.xyxy[:3])
    print('classes:', [VISDRONE_CLASSES[int(c)] for c in res.boxes.cls[:3]])
    print('scores:', res.boxes.conf[:3])

## 6. Real training at scale + Hub push

For a real run, download VisDrone (official site or an HF mirror) and use the fine-tune script:

```bash
python scripts/finetune_yolo9_visdrone.py \
    --visdrone-root /data/VisDrone \
    --size s --epochs 50 --batch 16 --imgsz 640 \
    --push --hf-repo ander2221/libreyolo-yolo9s-visdrone
```

Or with HF Hub–hosted dataset:

```bash
python scripts/finetune_yolo9_visdrone.py \
    --hf-dataset Voxel51/visdrone2019-det \
    --size s --epochs 50 --batch 16 --imgsz 640 \
    --push --hf-repo ander2221/libreyolo-yolo9s-visdrone
```

Expected wall-clock on an A10G: ~3 hours for 50 epochs at size s / imgsz 640. The script writes a model card with class names and usage snippet.

See `docs/agentic-features/blog/yolo9-visdrone-finetune.md` for the full story.